# Composing skills: handoff, shared skill, and the dependency graph

```yaml
title: "Composing skills: handoff, shared skill, and the dependency graph"
keywords: skill composition, sequential handoff, shared lower-level skill, seam contract, dependency graph, list_skills, load_skill, just-in-time loading, always-on cost, tool schemas, fan-in, pipeline, depth, over-chaining, create_agent, FakeModel
estimated duration: 40
```

<a id="top"></a>

> **Lesson:** L23 — Skill patterns & composition. Teacher demo notebook — the runnable
> core for **Objectives 3 & 4** (compose two ways; reason about the dependency graph and JIT
> across it). Read [L2301_intro.md](L2301_intro.md) and the archetypes lecture
> [L2302_lecture.md](L2302_lecture.md) first; the anti-patterns are next in
> [L2305_lecture.ipynb](L2305_lecture.ipynb).
> **Roadmap:** [objectives.md](../../../../docs/origin/lesson_roadmaps/L23/objectives.md).
> **Anchor model: Claude Sonnet 4.6.**
>
> **Runs two ways with the same code.** By default this notebook drives the offline scripted
> **`FakeModel`** (no key, deterministic, restart-and-run-all passes). Set `ANTHROPIC_API_KEY`
> and the optional live cells drive real **Sonnet 4.6**.
>
> The point to land: **composition is `load_skill` calls across a graph.** A *handoff* is one
> skill's body ending in `load_skill("next")`; a *shared skill* is two bodies both calling
> `load_skill("C")`. Both make a **dependency graph**, and the only always-on cost is the **two
> tool schemas** — bodies load just-in-time, only along the path taken.


## Contents

- [1. Setup — the skill registry and the two JIT tools](#1-setup--the-skill-registry-and-the-two-jit-tools)
- [2. Sequential handoff (A → B)](#2-sequential-handoff-a--b)
- [3. Shared lower-level skill (A → C ← B)](#3-shared-lower-level-skill-a--c--b)
- [4. The dependency graph](#4-the-dependency-graph)
- [5. Just-in-time loading across the graph, in numbers](#5-just-in-time-loading-across-the-graph-in-numbers)
- [6. Depth costs loads (setup for the anti-patterns)](#6-depth-costs-loads-setup-for-the-anti-patterns)
- [7. (Optional) the same agent on a live model](#7-optional-the-same-agent-on-a-live-model)
- [8. Takeaways](#8-takeaways)


## 1. Setup — the skill registry and the two JIT tools

This `REGISTRY` models this repo's **real** `.claude/skills/` system in miniature. Its three
skills are exactly the ones the demos read on screen: `author-lesson-roadmap` (stage 1, a
procedure-centric runbook), `generate-materials-from-roadmap` (stage 2, the downstream half of
the handoff), and `check-lesson-links` (a small shared skill both stages invoke). Each `Skill`
carries a `name`, a `description` (the trigger the model matches on — L22's craft point), and a
`body` of instructions.

The whole runtime is the **L11 `create_agent` shallow agent** given **two LangChain tools**
(`make_skill_tools`): `list_skills()` returns every skill's `{name, description}` (**discovery**)
and `load_skill(name)` reads one skill's full body into context (**load**). This is L22's
just-in-time mechanism expressed as *agent tools* rather than a hand-rolled loader. The bodies
above deliberately contain literal `load_skill("...")` calls — that is what makes *"skill A
invokes skill B"* concrete and, in Section 4, machine-readable.

The setup cell runs with no edits and no key: it drives the offline `FakeModel` by default and
only touches Sonnet 4.6 in the optional Section 7.


In [1]:
from __future__ import annotations

import re
from collections.abc import Callable
from dataclasses import dataclass
from typing import Annotated

from langchain.agents import create_agent
from langchain_core.messages import AIMessage, HumanMessage

from fluffy_potato_curriculum.common.config import get_settings
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)

# The course anchor model. ChatAnthropic wants the bare id (no 'anthropic:' prefix).
SONNET = "claude-sonnet-4-6"
# Live cells are gated on a present key, so a keyless run still completes.
LIVE = bool(get_settings().anthropic_api_key)


@dataclass(frozen=True)
class Skill:
    """A skill: a name, a description that says WHEN it applies, and a body."""

    name: str
    description: str
    body: str


# A tiny stand-in for this repo's real curriculum-authoring skill system. Bodies
# reference load_skill("X") so the dependency edges are literal, runnable text.
REGISTRY: dict[str, Skill] = {
    "author-lesson-roadmap": Skill(
        name="author-lesson-roadmap",
        description="Draft a lesson's roadmap docs (objectives, demos) from the PRD. Stage 1.",
        body=(
            "1. Read the PRD row and format specs.\n"
            "2. Draft objectives.md and demos_or_activities.md.\n"
            "3. Validate cross-references: load_skill('check-lesson-links').\n"
            "4. Hand off to stage 2: load_skill('generate-materials-from-roadmap')."
        ),
    ),
    "generate-materials-from-roadmap": Skill(
        name="generate-materials-from-roadmap",
        description="Generate student materials (lectures, labs) from a roadmap. Stage 2.",
        body=(
            "1. Read the roadmap docs produced by stage 1.\n"
            "2. Generate intro, lectures, and lab pairs.\n"
            "3. Validate cross-references: load_skill('check-lesson-links')."
        ),
    ),
    "check-lesson-links": Skill(
        name="check-lesson-links",
        description="Check one document's markdown links; judge each unresolved one. Shared.",
        body="Run extract_links.py, then classify each unresolved link (broken / placeholder).",
    ),
}


def make_skill_tools(registry: dict[str, Skill]) -> list[Callable[..., str]]:
    """Build the two JIT tools over a skill registry: discovery + load."""

    def list_skills() -> str:
        """Discovery: list every registered skill as `name: description`.

        Call this first to see what is available before choosing one to load.
        """
        return "\n".join(f"{s.name}: {s.description}" for s in registry.values())

    def load_skill(
        name: Annotated[str, "the exact skill name from list_skills to read into context"],
    ) -> str:
        """Load: read one skill's full body into context, or an error if unknown."""
        skill = registry.get(name)
        if skill is None:
            return f"no such skill: {name!r}"
        return skill.body

    return [list_skills, load_skill]


def loaded_skills(messages: list[AIMessage]) -> list[str]:
    """Which skills the agent actually pulled, in order, from the run history."""
    return [
        call["args"]["name"]
        for msg in messages
        if isinstance(msg, AIMessage)
        for call in msg.tool_calls
        if call["name"] == "load_skill"
    ]


def dependency_edges(registry: dict[str, Skill]) -> list[tuple[str, str]]:
    """Extract 'A -> B' edges: skill A's body that calls load_skill('B')."""
    pattern = re.compile(r"load_skill\(['\"]([^'\"]+)['\"]\)")
    edges: list[tuple[str, str]] = []
    for skill in registry.values():
        for target in pattern.findall(skill.body):
            edges.append((skill.name, target))
    return edges


# The system prompt for every run below: discover, then load, then follow.
SYSTEM = (
    "You are the curriculum-build agent. Call list_skills() to discover skills, "
    "then load_skill(name) to read the one you need and follow it."
)

# Print the registry catalog and what list_skills() surfaces (they are the same
# {name: description} pairs -- discovery is exactly the always-on-in-L22 layer,
# now behind a tool call).
print("registry (name -> description):")
for skill in REGISTRY.values():
    print(f"  {skill.name}: {skill.description}")

print("\nlist_skills() returns:\n")
list_skills, load_skill = make_skill_tools(REGISTRY)
print(list_skills())
print("\nlive model:", SONNET if LIVE else "(no ANTHROPIC_API_KEY -- live section skips)")

/Users/timlee/myrepos/fluffy-potato-curriculum/.claude/worktrees/l23-stage2-materials/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


registry (name -> description):
  author-lesson-roadmap: Draft a lesson's roadmap docs (objectives, demos) from the PRD. Stage 1.
  generate-materials-from-roadmap: Generate student materials (lectures, labs) from a roadmap. Stage 2.
  check-lesson-links: Check one document's markdown links; judge each unresolved one. Shared.

list_skills() returns:

author-lesson-roadmap: Draft a lesson's roadmap docs (objectives, demos) from the PRD. Stage 1.
generate-materials-from-roadmap: Generate student materials (lectures, labs) from a roadmap. Stage 2.
check-lesson-links: Check one document's markdown links; judge each unresolved one. Shared.

live model: (no ANTHROPIC_API_KEY -- live section skips)


[↑ Back to top](#top)

## 2. Sequential handoff (A → B)

The first composition shape is **sequential handoff**: one skill produces an artifact the next
consumes, and the *pipeline* is the capability. This repo runs on a real one — the two-stage
authoring flow that built the lesson you are reading:

- **`author-lesson-roadmap`** (stage 1) writes the roadmap docs — `objectives.md`,
  `demos_or_activities.md` — under `docs/origin/lesson_roadmaps/L<NN>/`.
- **`generate-materials-from-roadmap`** (stage 2) reads those docs and produces the student
  materials (intros, lectures, labs).

**Stage 1's output is stage 2's input.** The **seam / handoff contract** is those roadmap `.md`
files: the agreed artifact stage 1 must produce for stage 2 to run at all. That stable seam is
*why two skills beats one mega-skill* — each half is separately **loadable** (JIT-loaded only for
the stage you are in), **testable**, and **re-runnable** (regenerate materials without re-drafting
the roadmap).

On the runtime, the arrow **is** a `load_skill` call: `author-lesson-roadmap`'s body ends with
`load_skill('generate-materials-from-roadmap')`. Below, the scripted agent walks that edge —
`list_skills` to discover, then the two `load_skill` calls, in order.


In [2]:
# Script a run that walks stage 1 -> stage 2: discover, load author-lesson-roadmap
# (whose body ENDS by telling the agent to load the stage-2 skill), then load
# generate-materials-from-roadmap, then answer. The final load_skill IS the handoff.
handoff_model = FakeModel(
    [
        tool_reply(tool_call("c1", "list_skills", {})),
        tool_reply(tool_call("c2", "load_skill", {"name": "author-lesson-roadmap"})),
        tool_reply(tool_call("c3", "load_skill", {"name": "generate-materials-from-roadmap"})),
        text_reply("Roadmap drafted (stage 1), then materials generated from it (stage 2)."),
    ]
)
handoff_agent = create_agent(handoff_model, make_skill_tools(REGISTRY), system_prompt=SYSTEM)
handoff_messages = handoff_agent.invoke(
    {"messages": [HumanMessage(content="Build lesson L99 end to end.")]}
)["messages"]

# The path the agent actually walked, read off its own history.
path = loaded_skills(handoff_messages)
print("loaded path:", path)
print("A -> B pipeline:", " -> ".join(path))
print("\nfinal:", handoff_messages[-1].content)

loaded path: ['author-lesson-roadmap', 'generate-materials-from-roadmap']
A -> B pipeline: author-lesson-roadmap -> generate-materials-from-roadmap

final: Roadmap drafted (stage 1), then materials generated from it (stage 2).


[↑ Back to top](#top)

## 3. Shared lower-level skill (A → C ← B)

The second shape is a **shared lower-level skill**: one small skill that *several* higher-level
"operating" skills all invoke, so a common capability is written **once and reused many** times.
Here `check-lesson-links` is that node — both `author-lesson-roadmap` (checking cross-refs in a
fresh roadmap) and `generate-materials-from-roadmap` (checking them in generated materials) call
it rather than each re-implementing the check.

Contrast the two shapes:

- **Section 2 was `A → B`** — output-to-input, a *pipeline*.
- **This is `A → C ← B`** — two callers, one *shared dependency*.

Both create a graph; they create different *shapes*. The run below walks the full system —
author → check → generate → check — and the key thing to watch is that `check-lesson-links`
loads **per use, not per caller**: it appears twice in the path because it was *used* twice (once
inside each operating skill), not because there are two copies of it.


In [3]:
# Script the FULL walk: author -> check -> generate -> check. BOTH operating skills
# (author, generate) load check-lesson-links. The shared skill loads PER USE, not
# per caller: it appears twice in the path because it was used twice, once inside
# each operating skill.
shared_model = FakeModel(
    [
        tool_reply(tool_call("c1", "list_skills", {})),
        tool_reply(tool_call("c2", "load_skill", {"name": "author-lesson-roadmap"})),
        tool_reply(tool_call("c3", "load_skill", {"name": "check-lesson-links"})),
        tool_reply(tool_call("c4", "load_skill", {"name": "generate-materials-from-roadmap"})),
        tool_reply(tool_call("c5", "load_skill", {"name": "check-lesson-links"})),
        text_reply("Roadmap drafted, links checked, materials generated, links checked again."),
    ]
)
shared_agent = create_agent(shared_model, make_skill_tools(REGISTRY), system_prompt=SYSTEM)
shared_messages = shared_agent.invoke(
    {"messages": [HumanMessage(content="Build lesson L99, validating links at each stage.")]}
)["messages"]

shared_path = loaded_skills(shared_messages)
print("loaded path:", shared_path)
uses = shared_path.count("check-lesson-links")
print("\ncheck-lesson-links loaded", uses, "times -- once per USE, not per caller")
print("(A -> C ..... B -> C: two callers, one shared dependency node C)")

loaded path: ['author-lesson-roadmap', 'check-lesson-links', 'generate-materials-from-roadmap', 'check-lesson-links']

check-lesson-links loaded 2 times -- once per USE, not per caller
(A -> C ..... B -> C: two callers, one shared dependency node C)


[↑ Back to top](#top)

## 4. The dependency graph

Once skills call skills, there is a **skill dependency graph**: **nodes are skills**, **edges are
`load_skill` calls**. It is the *map of possible loads* — every `load_skill("X")` sitting in some
skill's body is a potential edge the agent may walk.

Because the bodies contain literal `load_skill("...")` text, we can read the edges straight out of
the registry with a regex — `dependency_edges` does exactly that. From the edges we can name each
structural piece: the **pipeline** (the `author → generate` handoff), the **fan-in / shared node**
(`check-lesson-links`, with more than one caller), and the **leaves** (skills that load nothing
further).


In [4]:
# The graph is the map of POSSIBLE load_skill calls: every load_skill("X") sitting
# in some skill's body is an edge. dependency_edges reads them straight out of the
# bodies with a regex, so the edges are literal, runnable text.
edges = dependency_edges(REGISTRY)
print("edges (A -> B means A's body calls load_skill('B')):")
for source, target in edges:
    print(f"  {source} -> {target}")

# Classify the pieces by their graph role.
sources = {s for s, _ in edges}
fan_in = [name for name in REGISTRY if sum(t == name for _, t in edges) > 1]
leaves = [name for name in REGISTRY if name not in sources]

print(
    "\npipeline edge (handoff):",
    [(s, t) for s, t in edges if t == "generate-materials-from-roadmap"],
)
print("shared / fan-in node (>1 caller):", fan_in)
print("leaves (load nothing further):", leaves)

edges (A -> B means A's body calls load_skill('B')):
  author-lesson-roadmap -> check-lesson-links
  author-lesson-roadmap -> generate-materials-from-roadmap
  generate-materials-from-roadmap -> check-lesson-links

pipeline edge (handoff): [('author-lesson-roadmap', 'generate-materials-from-roadmap')]
shared / fan-in node (>1 caller): ['check-lesson-links']
leaves (load nothing further): ['check-lesson-links']


Drawn as a graph, the three skills and their `load_skill` edges look like this. Two
operating skills (`author-lesson-roadmap`, `generate-materials-from-roadmap`) sit on a
**pipeline** (the handoff), and both **fan in** to the one shared node
(`check-lesson-links`):

```mermaid
graph LR
    A["author-lesson-roadmap<br/>(stage 1 / runbook)"]
    B["generate-materials-from-roadmap<br/>(stage 2 / runbook)"]
    C["check-lesson-links<br/>(shared / leaf)"]
    A -->|handoff| B
    A -->|shared| C
    B -->|shared| C
```

- **Pipeline (chain):** `A → B` — the sequential handoff from Section 2.
- **Fan-in (shared node):** `A → C` and `B → C` — the shared dependency from Section 3.
- **Leaf:** `C` loads nothing further — the bottom of this little system.

The graph is a **map of possible loads, not guaranteed control flow.** Every edge is a
*model-driven* `list_skills` → `load_skill` selection that can miss; the agent walks
only the edges the task actually needs.


[↑ Back to top](#top)

## 5. Just-in-time loading across the graph, in numbers

Now the cost lens (L01 / L07), scaled from one skill to the whole graph. The crucial refinement
of **L22**: in L22 every skill's `name + description` sat **always-on** in the system prompt. Here
routing discovery through `list_skills` pushes even *that* to just-in-time — so the only always-on
skill cost is the **two tool schemas**, no matter how many skills exist or what shape the graph is.

Three layers, three billing rules:

- **Always-on:** the two tool schemas (`list_skills`, `load_skill`). Fixed — does *not* grow with
  the registry.
- **Discovery:** the `{name: description}` catalog, paid **per `list_skills` call** (this is L22's
  always-on layer, now just-in-time).
- **Bodies:** paid **per `load_skill`** along the path taken; a shared body counts **once** in
  context even if loaded twice, and an unused skill's body is **never** paid.


In [5]:
# ALWAYS-ON: not the skills -- just the TWO tool schemas. It does not grow with the
# registry: three skills or three hundred, the always-on skill cost is these two tools.
always_on_tools = ["list_skills", "load_skill"]
print("always-on (regardless of how many skills exist):", always_on_tools)
print("  ->", len(always_on_tools), "tool schemas, and nothing else")

# DISCOVERY: paid per list_skills() call -- the {name: description} catalog. In L22
# this layer sat always-on; routing it through a tool makes even discovery just-in-time.
discovery_chars = len(list_skills())
print("\ndiscovery cost (per list_skills call):", discovery_chars, "chars of catalog")

# BODIES: paid per load_skill along the path taken. The shared body is counted ONCE
# even though it loaded twice -- distinct bodies are what sit in context.
distinct_bodies = list(dict.fromkeys(shared_path))
body_chars = sum(len(REGISTRY[name].body) for name in distinct_bodies)
print("\ndistinct bodies loaded on the Section 3 path:", distinct_bodies)
print("body cost (unique bodies on the path):", body_chars, "chars")

# The contrast that refines L22: you only pay for the bodies on the path -- an unused
# skill's body never enters context at all.
print("\nunused skills pay NO body cost; only the path is billed.")

always-on (regardless of how many skills exist): ['list_skills', 'load_skill']
  -> 2 tool schemas, and nothing else

discovery cost (per list_skills call): 289 chars of catalog

distinct bodies loaded on the Section 3 path: ['author-lesson-roadmap', 'check-lesson-links', 'generate-materials-from-roadmap']
body cost (unique bodies on the path): 456 chars

unused skills pay NO body cost; only the path is billed.


[↑ Back to top](#top)

## 6. Depth costs loads (setup for the anti-patterns)

Reasoning about the graph's *shape* comes down to one trade-off: **each hop is an extra load, and
often an extra model turn.** A handoff adds a `load_skill`; a level of sub-skill nesting adds
another. That has a clear consequence:

- **Wide-and-shallow** graphs — many independent skills, few hops — load **cheaply** and **fail
  locally**: a miss touches one skill, not the whole run.
- **Deep chains** — long handoff pipelines, sub-skills calling sub-skills — **concentrate risk**:
  each hop is another model-driven `list_skills` → `load_skill` *selection* that can miss, and a
  miss *anywhere* breaks the pipeline. More loads, more turns, more places to go wrong.

This is exactly the setup for the first composition anti-pattern, **over-chaining** — slicing a
task into so many tiny handoffs that the pipeline is mostly load-overhead and every hop is a place
selection can fail. That, plus a shared "skill" that's really a *tool* and **description
collisions**, is the subject of the next notebook: [L2305_lecture.ipynb](L2305_lecture.ipynb).

> Two forward pointers, not taught here: keeping a long-running agent's context lean across many
> loads is L19 (context management, full course); orchestrating whole *agents* this way — not
> skills — is L24 (multi-agent, full course).


[↑ Back to top](#top)

## 7. (Optional) the same agent on a live model

Swap the `FakeModel` for a real `ChatAnthropic` (Sonnet 4.6) and the **same** `create_agent` call
drives a live model over the **same** two tools — now the model *itself* decides which skills to
`list_skills` and `load_skill`. A real model varies, so the exact path can differ run to run; the
point is that the *code* is identical. This cell is gated on a key: with none set it prints a skip
line so a keyless run still passes.

**Clear this cell's output before committing** — live output is non-deterministic.


In [6]:
if LIVE:
    from langchain_anthropic import ChatAnthropic

    live_model = ChatAnthropic(model=SONNET)  # reads ANTHROPIC_API_KEY from the environment
    live_agent = create_agent(live_model, make_skill_tools(REGISTRY), system_prompt=SYSTEM)
    live_messages = live_agent.invoke(
        {"messages": [HumanMessage(content="Build lesson L99 end to end.")]}
    )["messages"]
    print("live loaded path:", loaded_skills(live_messages))
    print("live final:", live_messages[-1].content)
else:
    print("No ANTHROPIC_API_KEY set -- skipping the live run.")
    print("Set it in your environment or repo-root .env to drive Sonnet 4.6.")

No ANTHROPIC_API_KEY set -- skipping the live run.
Set it in your environment or repo-root .env to drive Sonnet 4.6.


[↑ Back to top](#top)

## 8. Takeaways

- **The seam is the design.** A handoff is worth it only when the boundary between two skills is a
  real, stable *contract* (a well-defined artifact) — that is what makes each half separately
  loadable, testable, and re-runnable. The repo's `author-lesson-roadmap →
  generate-materials-from-roadmap` flow is a real one; the roadmap docs are the seam.
- **Two composition shapes, both a graph.** Sequential handoff is `A → B` (a pipeline); a shared
  lower-level skill is `A → C ← B` (a fan-in dependency). Same registry, different graph shape.
- **An edge is a `load_skill` call.** The dependency graph is the *map of possible loads*, read
  literally off the skill bodies — not guaranteed control flow. Every edge is a model-driven
  selection that can miss.
- **Always-on cost = two tool schemas.** Regardless of how many skills exist or the graph's shape.
  Discovery is paid per `list_skills`; bodies per `load_skill` along the path; a shared body loads
  per-use but counts once in context; unused skills cost nothing. This *refines* L22, where the
  descriptions sat always-on.
- **Depth costs loads.** Wide-and-shallow loads cheaply and fails locally; deep chains concentrate
  risk. That sets up **over-chaining** and the other anti-patterns in
  [L2305_lecture.ipynb](L2305_lecture.ipynb).


[↑ Back to top](#top)